<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_01_fdm_ground_truth.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 01 — The Ground Truth, by Finite Differences

**Paired with L7.1 · Fundamentals of PINNs**

The slot from notebook 00, solved the classical way. Finite differences lay a
grid over the slot, replace every derivative by differences between
neighbouring nodes, and turn the equation into one linear system — which a
computer solves in milliseconds.

This notebook produces the answer the rest of the set is scored against, so it
has to earn that job. You show that the answer stops changing as the grid gets
finer, and then you make it as accurate as the method allows.

## What you will do

1. Write the heat source: the 32 wires' $RI^2$, with copper's resistance rising
   as it warms.
2. Solve the slot at rated current, and find its hot spot.
3. Refine the grid until the answer stops changing, and time every solve.
4. Combine two grids into a ground truth more accurate than either.
5. Watch the temperature rise grow faster than $I^2$, and save what notebook 02
   needs.

## One thing to watch

A reference has to be far more accurate than what it judges. The networks in
notebook 02 get the hot spot to within a tenth of a kelvin or two; the ground
truth built here is good to about a hundred-thousandth. Only then is the error
you measure in notebook 02 the network's, and not the reference's.

---

## 0 · Setup

**What these cells do.** Fetch the library files, keep the results in your Google Drive so the later notebooks can read them, and import the modules.

**How.** The first cell downloads `course_core.py`, `pinn_core.py` and `problem.py` on Colab; the second asks for Drive access; the third imports them and checks they are really there.

In [ ]:
# files-cell v2 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository on every run, so a file left
# over from an earlier session cannot shadow it. Run this cell first.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    # on Colab always re-fetch: /content outlives a session and a stale copy
    # would silently shadow an updated one. Locally the repo's own files stay.
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
# a module imported before this cell would keep the previous file's contents
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import course_core as cc
cc.keep_outputs("Ex07.1_outputs")

In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)  # fix the random start so a run repeats
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

---

## 1 · The heat source

**What this cell does.** Defines $q(\theta, I)$, the heat the winding makes per cubic metre when the slot is $\theta$ kelvin above the wall and each wire carries $I$ amperes.

**How.** One metre of wire has resistance $\rho/A_{\text{wire}}$ and makes $RI^2$ of heat. Thirty-two of them, spread over the slot's cross-section, make

$$q = \text{fill}\cdot\rho(T)\,J^2, \qquad J = \frac{I}{A_{\text{wire}}}, \qquad \rho(T) = \rho_{20}\,[1 + \alpha\,(T - 20)], \qquad T = 90 + \theta$$

with `pb.FILL`, `pb.A_WIRE`, `pb.RHO_CU`, `pb.ALPHA_CU` and `pb.T_WALL` for the constants. The cell after it compares yours with `pb.heat_source`.

In [ ]:
# TODO 1: write the heat source q(theta, I), in W/m^3.
#
#   def heat_source(theta, I):
#       J = I / pb.A_WIRE                                                # A/m^2
#       rho = pb.RHO_CU * (1 + pb.ALPHA_CU * (pb.T_WALL + theta - 20))   # at T = 90 C + theta
#       return pb.FILL * rho * J ** 2                                    # 32 wires, over the slot
#
# The copper is at the wall temperature PLUS the rise. Leave out pb.T_WALL and
# its resistance is worked out at theta degrees - some ninety kelvin too cold.

raise NotImplementedError("Write heat_source(theta, I)")

In [ ]:
theta_test = np.array([0.0, 10.0, 40.0])        # kelvin above the wall
for I in (pb.I_RANGE[0], pb.I_RATED, pb.I_RANGE[1]):
    check(f"heat_source at {I:.1f} A", heat_source(theta_test, I) / pb.heat_source(theta_test, I),
          np.ones(3), tol=1e-12)

**What you should see.** Three `PASS` lines.

From here on the notebook uses `pb.heat_source`, the same function, so that
notebooks 02 and 03 compute exactly the heat this one does.

---

## 2 · One solve at rated current

**What this cell does.** Solves the slot at rated current on a grid of 41 × 81 nodes, finds the hot spot, and draws the temperature beside the grid that produced it.

**How.** At every interior node the five-point stencil stands in for the Laplacian,

$$\nabla^2\theta \approx \frac{\theta_E + \theta_W - 2\theta_P}{h^2} + \frac{\theta_N + \theta_S - 2\theta_P}{h^2},$$

using the node's four neighbours. The walls are $\theta = 0$ and are not unknowns. The heat is $q = c_0 + c_1\theta$, so every node gives one linear equation and the whole slot is one sparse system, $(k_{\text{eff}}L + c_1)\,\theta = -c_0$. `pb.fdm_matrix` builds it; `pb.fdm_solve` builds it and solves it. The left panel draws a grid small enough to count, 7 nodes across: the wall nodes are known, every node inside is one unknown, and one node is shown with the four neighbours its equation uses. The real solve does the same on 41 × 81 nodes; the note under the figure explains both panels.

In [ ]:
X, Y, theta = pb.fdm_solve(pb.I_RATED, nx=41)          # 41 x 81 nodes, walls included

i = np.unravel_index(theta.argmax(), theta.shape)
print(f"hot spot: {theta.max():.3f} K above the wall, {pb.T_WALL + theta.max():.1f} C, "
      f"at x = {X[i]*1e3:+.2f} mm, y = {Y[i]*1e3:+.2f} mm")
print(f"unknowns: {(41 - 2) * (81 - 2)}, each tied to its four neighbours only")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 5.8), gridspec_kw=dict(width_ratios=[1, 1]))
pb.plot_fdm_grid(7, ax=axes[0])                          # the method, on a grid you can count
pb.plot_field(theta.ravel(), 41, 81, ax=axes[1], wires=True,  # the answer, wires outlined
              title="temperature rise at rated current")
plt.tight_layout(); plt.show()

**What you should see.** A hot spot of **18.74 K** at the centre of the slot —
108.7 °C in the copper — and a field symmetric left to right and top to bottom.

**The left panel is the method, not the solve.** It draws the same kind of grid
the solver uses, only coarse enough to count: 7 nodes across the 10 mm width
and 13 along the 20 mm depth, both $h = 10/6 = 1.67$ mm apart. The depth gets
$2 \times 7 - 1 = 13$ nodes because the slot is twice as deep as it is wide and
the spacing is the same in both directions. The 20 squares on the wall are
known, $\theta = 0$; the 5 × 11 = 55 dots inside are the unknowns. The real
solve on the right uses the same rule with 41 × 81 nodes — too many to draw.

**The cross is the five-point stencil**, drawn at one node chosen as an
example. The red node P and its four neighbours, north, south, east and west,
$h$ away, are all its equation uses:

$$k_{\text{eff}}\,\frac{\theta_N + \theta_S + \theta_E + \theta_W - 4\theta_P}{h^2} + q(\theta_P) = 0 .$$

Every dot inside has the same cross around it, and one such equation each.
That this node sits on the edge of a wire means nothing: the grid does not
follow the wires. The winding is treated as one material with its heat spread
evenly (notebook 00), so a node inside a wire, on its edge or between two wires
gets exactly the same equation. The pale circles behind the grid are there only
to show where the copper is.

The white circles on the right are the same 32 wires, drawn on top of the
answer. The field does not show them either, for the same reason. Each unknown
is tied only to its four neighbours, so each equation has at most five terms,
which is why a system of three thousand unknowns solves in milliseconds.

---

## 3 · Has it converged?

**What this cell does.** Solves the same slot on six grids, each twice as fine as the one before, prints the hot spot from each, and times every solve.

**How.** For each grid it solves once to warm up — the first call pays one-off start-up costs — then three more times, and keeps the middle time. Notebook 02 compares these times with a trained network's, so they have to be honest: they include building the grid and the matrix, not only solving it.

In [ ]:
GRIDS = [11, 21, 41, 81, 161, 321]
sols, times = {}, {}
for nx in GRIDS:
    pb.fdm_solve(pb.I_RATED, nx)                           # warm-up, not timed
    ts = []
    for _ in range(3):
        t0 = time.perf_counter()
        _, _, th = pb.fdm_solve(pb.I_RATED, nx)            # grid + matrix + solve
        ts.append(time.perf_counter() - t0)
    sols[nx], times[nx] = th, float(np.median(ts))
    print(f"  {nx:4d} x {2*nx - 1:4d} grid, {(nx-2)*(2*nx-3):6d} unknowns: "
          f"hot spot {th.max():.5f} K   {times[nx]*1e3:8.1f} ms")

**What you should see.** The hot spot settling as the grid gets finer — the
same number to more and more digits — while the time grows roughly with the
number of unknowns. By the 41 × 81 grid the hot spot no longer changes in the
second decimal.

"Stops changing" is not yet "correct", though. To say how far each grid is
from the truth, you need the truth.

---

## 4 · A ground truth better than any one grid

**What this cell does.** Solves once more on a grid twice as fine as the finest above, and combines the two into a ground truth more accurate than either.

**How.** The stencil's error falls as $h^2$: halve the spacing and the error falls four times. Write the two solutions as truth $+\,c\,h^2$ and truth $+\,4c\,h^2$; subtract to eliminate $c$, and

$$\theta_{\text{truth}} \approx \frac{4\,\theta_{h} - \theta_{2h}}{3}.$$

This is **Richardson extrapolation**. `fine[::2, ::2]` picks the nodes of the fine grid that sit on the coarse one.

In [ ]:
# TODO 2: combine the two grids into the ground truth.
#
#   _, _, fine = pb.fdm_solve(pb.I_RATED, 641)               # 641 x 1281 nodes
#   coarse = sols[321]
#   truth = (4 * fine[::2, ::2] - coarse) / 3                # the h^2 error cancelled
#
# fine[::2, ::2] keeps every other node of the fine grid: exactly the nodes
# that sit on the 321 grid, so the two arrays line up.

raise NotImplementedError("Write truth = (4 * fine - coarse) / 3")

In [ ]:
check("ground truth, against problem.py", truth, pb.ground_truth(pb.I_RATED)[2], tol=1e-9)
print(f"hot spot {truth.max():.5f} K; the 641 grid alone is off by "
      f"{np.abs(fine[::2, ::2] - truth).max():.1e} K")

errs = {}
for nx in GRIDS:
    s = (321 - 1) // (nx - 1)                             # the truth's nodes that sit on this grid
    errs[nx] = float(np.abs(sols[nx] - truth[::s, ::s]).max())
    print(f"  {nx:4d} x {2*nx - 1:4d} grid: worst error {errs[nx]:.2e} K   {times[nx]*1e3:8.1f} ms")

h = np.array([2 * pb.A_HALF / (nx - 1) * 1e3 for nx in GRIDS])     # node spacing, mm
e = np.array([errs[nx] for nx in GRIDS])
fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.loglog(h, e, "o-", color=CYCLE[0], lw=2, label="finite differences")
ax.loglog(h, 2 * e[0] * (h / h[0]) ** 2, ":", color="0.4",
          label="slope 2, raised for comparison")
ax.set_xlabel("node spacing $h$  [mm]"); ax.set_ylabel("worst error  [K]")
ax.set_title("each halving of $h$ cuts the error four times")
ax.legend(frameon=False); plt.show()

**What you should see.** One `PASS` line, and a ground truth the 641 grid on its
own already agrees with to about $10^{-5}$ K. The errors fall four times with
each halving of $h$, so the points lie on the slope-2 line.

Read the table as an engineer would. **The 41 × 81 grid gets the hot spot to
within a few thousandths of a kelvin in a few milliseconds** — more accurate
than any thermocouple — and that is the grid and the time notebook 02 has to
compete with.

---

## 5 · Faster than $I^2$

**What this cell does.** Solves the slot at fifteen currents from 10 to 45 A and compares each hot spot with what pure $I^2$ scaling from rated current would predict.

**How.** If the heat were fixed, the rise would grow exactly as $I^2$ — double the current, four times the rise. Here hotter copper makes more heat, so the rise should grow faster. Each current is one `fdm_solve` on the 81 × 161 grid.

In [ ]:
currents = np.linspace(*pb.I_RANGE, 15)
peaks = np.array([pb.fdm_solve(I, 81)[2].max() for I in currents])
at_rated = pb.fdm_solve(pb.I_RATED, 81)[2].max()
scaled = at_rated * (currents / pb.I_RATED) ** 2        # what fixed heat would give

fig, ax = plt.subplots(figsize=(6.6, 4.2))
ax.plot(currents, peaks, "o-", color=CYCLE[0], lw=2, label="finite differences")
ax.plot(currents, scaled, ":", color="0.4", lw=2, label="$I^2$ scaling from rated")
ax.axvline(pb.I_RATED, color="0.7", lw=1)
ax.set_xlabel("current per wire  [A]"); ax.set_ylabel("hot spot, above the wall  [K]")
ax.set_title("the rise grows faster than $I^2$")
ax.legend(frameon=False); plt.show()
print(f"at {currents[-1]:.0f} A: {peaks[-1]:.2f} K, against {scaled[-1]:.2f} K from I^2 scaling "
      f"({(peaks[-1] / scaled[-1] - 1) * 100:+.1f} %)")

**What you should see.** The two curves meeting at rated current and parting
above it: at 45 A the hot spot is about **5 % higher** than $I^2$ scaling
predicts. Below rated it is lower, for the same reason run backwards — cooler
copper makes less heat.

That 5 % is copper's resistance rising with temperature. It matters for the
rest of the set: one solve at one current no longer gives you every other
current by rescaling, so a method that answers *any* current has something real
to offer. Notebook 02 finds out whether a network is that method, and
notebook 03 explains where the extra few per cent come from.

---

## 6 · Save

**What this cell does.** Builds a ground truth at six currents across the range and saves it, with the grids' errors and times, for notebook 02 and the report.

**How.** `pb.ground_truth(I, nx=161)` does the Richardson step of section 4 on the 161 and 321 grids — accurate to about $10^{-5}$ K and quicker than the 641 grid. The fields go into `nb01_truth.npz` in `Ex07.1_outputs/`.

In [ ]:
TEST_I = np.array([10.0, 17.5, 25.0, pb.I_RATED, 38.0, 45.0])
fields = np.stack([pb.ground_truth(I, nx=161)[2] for I in TEST_I])    # each 321 x 161

os.makedirs(cc.OUTPUT_DIR, exist_ok=True)
path = os.path.join(cc.OUTPUT_DIR, "nb01_truth.npz")
np.savez(path, currents=TEST_I, fields=fields, nx=161,
         grids=np.array(GRIDS), times=np.array([times[n] for n in GRIDS]),
         errors=np.array([errs[n] for n in GRIDS]))
print("wrote", path, "-", fields.shape[0], "currents, hot spots",
      ", ".join(f"{f.max():.2f}" for f in fields), "K")
cc.saved(path)

**What you should see.** Six hot spots, from under 2 K at 10 A to about 40 K at
45 A, and the path of the saved file.

---

## 7 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Finite differences turned the slot into one linear system. What are its unknowns, what did the grid do that a physics-informed network does without, and how many unknowns did your finest grid have?
   *→ L7.1 Q4*
2. The error fell four times with every halving of $h$. Say why, and how Richardson extrapolation uses that to build a reference better than either grid. Why must a reference be far more accurate than what it judges?
   *→ L7.1 Q4*
3. The hot spot grew faster than $I^2$. Explain why from the heat source, and why each current is nevertheless a single linear solve. What makes the problem well posed at 45 A, and what would stop it being so?
   *→ L7.1 Q5*
4. Which grid would you choose to know the hot spot to 0.01 K, and how long did it take? Keep that number: it is what notebook 02 has to beat.
   *→ L7.1 Q4*

*Write your answers here. You will copy them into the report in notebook 04, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 02](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard.ipynb)**, where a network learns the slot for every current at once, with its walls imposed softly and then hard, and is scored against this ground truth on accuracy and time.

<!-- two-forms -->
**Open notebook 02 as**

- **[the exercise](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard.ipynb)**, where you write the missing lines, or
- **[the light version](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_02_pinn_soft_and_hard_light.ipynb)**, with every cell written out.